# Notebook 3 — Pipeline Automatizado de Monitoramento de Drift

## Aula 2: Detecção Estatística de Drift de Dados

### Objetivos
- Construir pipeline de detecção de drift usando os módulos `src/`
- Simular monitoramento contínuo em janelas temporais
- Gerar relatórios consolidados de drift por feature
- Discutir boas práticas de MLOps para monitoramento de drift

### Teoria-Chave (Documento 04)

> "Em sistemas de ML em produção, é ideal implementar um monitoramento
> contínuo do data drift — parte essencial do MLOps. Isso significa calcular
> periodicamente métricas de drift conforme novos dados chegam."
>
> — Seção *Monitoramento Contínuo e MLOps*, Documento da Aula 2

### Vídeo Relacionado
**Vídeo 4** — Pipeline Automatizado de Monitoramento de Drift (18 min):  
Construção de pipeline com Evidently e NannyML; relatórios periódicos;
integração de alertas automáticos; boas práticas de MLOps.

In [ ]:
# Imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona diretório raiz da aula ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor
from src.model import DriftDetector, DriftReport
from src.training import (
    run_drift_detection,
    run_full_analysis,
    cross_validate_detection,
)
from src.evaluation import (
    plot_distribution_comparison,
    plot_psi_bars,
    plot_drift_heatmap,
    generate_drift_summary,
    calculate_metrics,
)
from src.utils import save_metrics, ensure_dirs

sns.set_theme(style="whitegrid")
%matplotlib inline

# Criar diretórios de saída
ensure_dirs()

## 1. Carga dos Dados

In [ ]:
# Carregar dataset
preprocessor = DataPreprocessor(seed=42, n_samples=5000)
data_path = Path.cwd().parent / "data" / "raw" / "dataset.csv"

if not data_path.exists():
    preprocessor.generate_and_save(str(data_path))

ref_data, prod_data = preprocessor.load_data(str(data_path))
print(f"Referência: {len(ref_data)} | Produção: {len(prod_data)}")

## 2. Pipeline de Detecção — Execução Única

Executamos a pipeline simplificada que combina KS + PSI para features numéricas
e Qui-quadrado para categóricas.

In [ ]:
# Pipeline padrão: KS + PSI + Chi2
report = run_drift_detection(
    ref_data, prod_data,
    methods=["ks", "psi"],
    alpha=0.05,
    psi_threshold=0.25,
)

print(f"Features analisadas: {report.n_features_total}")
print(f"Features com drift:  {report.n_features_drifted}")
print(f"Features afetadas:   {report.drifted_features}")

# Métricas agregadas
metrics = calculate_metrics(report)
print(f"\n% features com drift: {metrics['pct_features_drifted']:.1f}%")
if 'avg_psi' in metrics:
    print(f"PSI médio: {metrics['avg_psi']:.4f}")
    print(f"PSI máximo: {metrics['max_psi']:.4f}")

## 3. Análise Completa Multi-Método

Conforme a estratégia do Documento: screening com métricas + confirmação com testes.

In [ ]:
# Análise com todos os métodos
reports = run_full_analysis(ref_data, prod_data)

print(f"{'Método':<10} {'Features c/ Drift':>20} {'Lista':>40}")
print("-" * 75)
for method, rep in reports.items():
    print(f"{method.upper():<10} {rep.n_features_drifted:>20} {str(rep.drifted_features):>40}")

## 4. Validação Cruzada da Detecção

Para reduzir falsos positivos, validamos a detecção com subsampling repetido.
Se o drift é real, deve ser detectado consistentemente.

In [ ]:
# Cross-validation da detecção para feature 'idade' (KS test)
cv_results = cross_validate_detection(
    ref_data["idade"].values,
    prod_data["idade"].values,
    n_splits=10,
    method="ks",
    feature_name="idade",
)

print("Validação cruzada do teste KS (feature 'idade'):")
print(f"  Taxa de detecção: {cv_results['detection_rate']:.0%}")
print(f"  Estatísticas KS: {[f'{s:.4f}' for s in cv_results['statistics']]}")
print(f"  p-valores: {[f'{p:.2e}' for p in cv_results['p_values']]}")

## 5. Simulação de Monitoramento Temporal

> "Implementar monitoramento contínuo do data drift — parte essencial do MLOps.
> Ferramentas como Evidently ou NannyML oferecem pipelines prontos para computar
> diversas medidas de drift de forma automatizada."
>
> — Seção *Monitoramento Contínuo e MLOps*, Documento da Aula 2

Simulamos um cenário de monitoramento em 5 janelas temporais, onde drift gradual
é injetado a partir da janela 3.

In [ ]:
# Simulação temporal: gerar dados com drift progressivo
detector = DriftDetector(alpha=0.05)
n_windows = 6
window_size = 1000
rng = np.random.RandomState(42)

# Dados de referência (estáveis)
ref_ages = rng.normal(40, 10, size=5000)  # idade média 40

temporal_log = []

for w in range(n_windows):
    # Drift progressivo: a partir da janela 3, média cai gradualmente
    if w < 2:
        shift = 0  # sem drift
    else:
        shift = (w - 1) * 2  # drift crescente

    current_ages = rng.normal(40 - shift, 10, size=window_size)

    ks_result = detector.ks_test(ref_ages, current_ages, feature_name="idade")
    psi_result = detector.psi(ref_ages, current_ages, feature_name="idade")

    temporal_log.append({
        "janela": w + 1,
        "média_atual": current_ages.mean(),
        "ks_stat": ks_result.statistic,
        "ks_pvalue": ks_result.p_value,
        "ks_drift": ks_result.drift_detected,
        "psi": psi_result.statistic,
        "psi_drift": psi_result.drift_detected,
    })

temporal_df = pd.DataFrame(temporal_log)
display(temporal_df)

In [ ]:
# Visualização da evolução temporal do drift
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# KS Statistic ao longo do tempo
axes[0].plot(temporal_df["janela"], temporal_df["ks_stat"], "o-", color="steelblue", linewidth=2)
axes[0].set_xlabel("Janela Temporal")
axes[0].set_ylabel("Estatística KS")
axes[0].set_title("Estatística KS ao Longo do Tempo", fontweight="bold")
axes[0].axhline(y=0.05, color="red", linestyle="--", alpha=0.5, label="Limiar")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PSI ao longo do tempo
colors = ["forestgreen" if v < 0.10 else "darkorange" if v < 0.25 else "firebrick"
          for v in temporal_df["psi"]]
axes[1].bar(temporal_df["janela"], temporal_df["psi"], color=colors, edgecolor="white")
axes[1].axhline(y=0.10, color="orange", linestyle="--", label="Moderado (0.10)")
axes[1].axhline(y=0.25, color="red", linestyle="--", label="Severo (0.25)")
axes[1].set_xlabel("Janela Temporal")
axes[1].set_ylabel("PSI")
axes[1].set_title("PSI ao Longo do Tempo", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

# Média da feature ao longo do tempo
axes[2].plot(temporal_df["janela"], temporal_df["média_atual"], "o-", color="darkorange", linewidth=2)
axes[2].axhline(y=40, color="steelblue", linestyle="--", alpha=0.7, label="Média referência")
axes[2].set_xlabel("Janela Temporal")
axes[2].set_ylabel("Média da Idade")
axes[2].set_title("Evolução da Média Temporal", fontweight="bold")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle("Monitoramento Contínuo de Drift (Simulação)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 6. Sistema de Alertas

> "Detectar o drift leve a ações efetivas. Uma vez soado o alarme estatístico,
> a equipe deve investigar as causas: Quais atributos mudaram? O desempenho do
> modelo já foi afetado?"
>
> — Seção *Monitoramento Contínuo e MLOps*, Documento da Aula 2

Implementamos um sistema simples de alertas baseado em limiares.

In [ ]:
# Sistema de alertas
print("=" * 60)
print("SISTEMA DE ALERTAS DE DRIFT")
print("=" * 60)

for _, row in temporal_df.iterrows():
    janela = int(row["janela"])
    psi_val = row["psi"]
    ks_drift = row["ks_drift"]

    if psi_val >= 0.25:
        severity = "🔴 CRÍTICO"
        action = "→ Ação imediata: avaliar retrain ou pausa do modelo"
    elif psi_val >= 0.10:
        severity = "🟡 ATENÇÃO"
        action = "→ Investigar causa e planejar atualização"
    else:
        severity = "🟢 OK"
        action = "→ Sem ação necessária"

    print(f"\nJanela {janela}: {severity}")
    print(f"  PSI = {psi_val:.4f} | KS drift = {ks_drift}")
    print(f"  {action}")

## 7. Salvando Resultados

Persistimos os resultados para rastreabilidade e auditoria.

In [ ]:
# Salvar métricas e relatório
output_dir = Path.cwd().parent / "outputs"

# Métricas agregadas
metrics_path = save_metrics(
    metrics, output_dir / "logs" / "drift_metrics.json"
)
print(f"Métricas salvas em: {metrics_path}")

# Relatório temporal
temporal_df.to_csv(output_dir / "logs" / "temporal_drift.csv", index=False)
print(f"Relatório temporal salvo em: {output_dir / 'logs' / 'temporal_drift.csv'}")

# Tabela resumo
summary_df = generate_drift_summary(report)
summary_df.to_csv(output_dir / "logs" / "drift_summary.csv", index=False)
print(f"Resumo salvo em: {output_dir / 'logs' / 'drift_summary.csv'}")

## 8. Boas Práticas de MLOps para Monitoramento

Conforme discutido no Vídeo 4 e na seção *Monitoramento Contínuo e MLOps*:

1. **Monitoramento periódico:** Calcular métricas de drift em intervalos regulares
2. **Alertas calibrados:** Definir limiares adequados ao contexto de negócio
3. **Ações escalonadas:** PSI < 0.10 → ok; 0.10–0.25 → atenção; > 0.25 → ação
4. **Investigação de causas:** Quais features mudaram? Por quê?
5. **Ferramentas open-source:** Evidently, NannyML para automação
6. **Rastreabilidade:** Salvar resultados para auditoria

In [ ]:
# Demonstração conceitual de um loop de monitoramento MLOps
def run_monitoring_cycle(
    reference_data: pd.DataFrame,
    production_batches: list[pd.DataFrame],
    psi_threshold: float = 0.25,
):
    """Simula um ciclo de monitoramento de drift em produção.

    Conforme recomendado na seção 'Monitoramento Contínuo e MLOps':
    calcular periodicamente métricas de drift e gerar alertas.
    """
    detector = DriftDetector(psi_threshold=psi_threshold)
    alerts = []

    for i, batch in enumerate(production_batches):
        report = detector.detect_all(reference_data, batch, methods=["ks", "psi"])

        if report.n_features_drifted > 0:
            alerts.append({
                "batch": i + 1,
                "features_drifted": report.n_features_drifted,
                "drifted_list": report.drifted_features,
                "action": "RETRAIN" if report.n_features_drifted > 3 else "MONITOR",
            })

    return alerts

# Executar ciclo de demonstração
batches = [prod_data.sample(n=500, random_state=i) for i in range(5)]
alerts = run_monitoring_cycle(ref_data, batches)

print(f"Total de alertas: {len(alerts)}")
for alert in alerts:
    print(f"  Batch {alert['batch']}: {alert['features_drifted']} features → {alert['action']}")

## Resumo da Aula 2

Ao longo dos 3 notebooks, abordamos o pipeline completo de detecção de drift:

### Notebook 1 — Exploração
- EDA visual do dataset de crédito
- Evidência visual de drift em idade, renda, score

### Notebook 2 — Implementação de Métricas
- KS test: teste não-paramétrico para distribuições contínuas
- PSI: métrica do setor bancário com limiares de alerta
- KL/JS divergências: métricas de informação
- Qui-quadrado: teste para variáveis categóricas

### Notebook 3 — Pipeline Automatizado
- Execução de pipeline multi-método
- Monitoramento temporal simulado
- Sistema de alertas com ações escalonadas
- Boas práticas de MLOps

### Estatística-Chave do Documento:
> "91% dos modelos de ML sofrem algum grau de drift durante seu ciclo de vida,
> e modelos mantidos sem atualizações por mais de 6 meses apresentaram aumentos
> médios de 35% na taxa de erro."
>
> — Seção *Monitoramento Contínuo e MLOps*, Documento da Aula 2

---

**Próxima Aula:** Aula 3 — Estratégias de Mitigação de Drift